# Optuna Recall 제약 탐색

안쪽, 바깥 Scaffold 5-Fold. 독성 Recall ≥ 0.87 조건에서 비독성 F1 최대화. F1 달성 목표는 0.85이며, 바깥 Fold에서 Recall이 보장되지는 않는다. SQLite와 sampler 상태를 저장해 중단 후 재개하고 trial 예산을 늘릴 수 있다.


In [ ]:
from pathlib import Path
import subprocess, sys, json
import pandas as pd

# Optuna를 설치한 프로젝트 환경을 사용한다.
PYTHON = Path('.venv/bin/python').resolve()
assert PYTHON.is_file(), 'requirements-training.txt를 설치한 Python 경로를 지정하세요.'
TRIALS = 20  # 바깥 Fold당 예산. --resume에서는 누적 총 예산이다.
RECALL_TARGET = 0.87
OUTPUT = Path('Data/training/optuna_recall87_v1')
RESUME = True  # 기존 실행을 이어가거나, 완료된 동일 예산 결과를 읽는다.
command = [str(PYTHON), '-u', '-m', 'analysis.optuna_recall', '--trials', str(TRIALS),
           '--recall-target', str(RECALL_TARGET), '--output', str(OUTPUT)]
subprocess.run(command + ['--dry-run'], check=True)


In [ ]:
# 같은 출력 폴더에서 다른 프로세스가 실행 중이면 동시에 실행하지 않는다.
subprocess.run(command + (['--resume'] if RESUME else []), check=True)


In [ ]:
display(pd.read_csv(OUTPUT / 'outer_metrics.csv'))
display(pd.read_csv(OUTPUT / 'baseline_comparison.csv'))
print(json.loads((OUTPUT / 'evaluation.json').read_text()))
